# 03 — Duplicate Check (completes Gate 1)
Read-only check for exact and near-duplicate images on the **frozen** split (PROJECT_SPEC §6, DATA_PROTOCOL duplicate policy, pre-training audit finding B1).

- **Exact duplicates:** SHA256 of the file bytes.
- **Near duplicates:** 256-bit difference hash (dHash) compared by Hamming distance. It catches re-saved / re-encoded copies and near-identical shots, **not** the same object photographed from a different angle.

Nothing here modifies images or split files. Outputs go to `04_Results/dataset_audit/`. The numbers only flag candidates: the team decides after looking at the review grid and records the decision in `docs/DECISION_LOG.md`.

Run after `02_split_generator.ipynb`. A CPU runtime is enough; hashing 4752 images from Drive takes a few minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
from src.utils import load_config, get_paths
from src.dataset import load_frozen_split

cfg = load_config(REPO_DIR / "configs" / "config.yaml")
paths = get_paths(cfg)

# Verifies both SHA256 fingerprints, the split counts, the class mapping, and that every image exists.
split_df = load_frozen_split(cfg)
print(split_df["split"].value_counts().to_dict())

In [ ]:
from src import duplicate_check as dc

HASH_SIZE = 16              # 16x16 = 256-bit dHash
NEAR_DUP_MAX_DISTANCE = 20  # candidate threshold (~8% of bits); heuristic, check the histogram below

sha256s, bits = dc.compute_hashes(split_df, hash_size=HASH_SIZE)
print("Hashed images:", len(sha256s))

In [ ]:
exact_groups = dc.exact_duplicate_groups(split_df, sha256s)
print("Exact-duplicate groups:", len(exact_groups))
print("  spanning more than one split:", int(exact_groups["cross_split"].sum()))
print("  spanning more than one class:", int(exact_groups["cross_class"].sum()))
exact_groups.head(20)

In [ ]:
import matplotlib.pyplot as plt

# One row per validation/test image: its closest training image.
nearest = dc.nearest_train_neighbors(split_df, bits)
n_close = int((nearest["distance"] <= NEAR_DUP_MAX_DISTANCE).sum())
print(f"val/test images with a training image within distance {NEAR_DUP_MAX_DISTANCE}: {n_close}")
print(nearest["distance"].describe().round(1).to_string())

plt.figure(figsize=(8, 3))
plt.hist(nearest["distance"], bins=range(0, int(nearest["distance"].max()) + 2))
plt.axvline(NEAR_DUP_MAX_DISTANCE, color="red", linestyle="--", label=f"threshold = {NEAR_DUP_MAX_DISTANCE}")
plt.xlabel("dHash distance to nearest training image")
plt.ylabel("val/test images")
plt.legend()
plt.tight_layout()
plt.savefig(paths.audit_dir / "nearest_train_distance_hist.png", dpi=120)
plt.show()

In [ ]:
near_pairs = dc.near_duplicate_pairs(split_df, bits, NEAR_DUP_MAX_DISTANCE)
print("Near-duplicate candidate pairs (any splits):", len(near_pairs))
print("  across splits:", int(near_pairs["cross_split"].sum()))
print("  across classes:", int(near_pairs["cross_class"].sum()))
near_pairs.head(20)

In [ ]:
from PIL import Image

# Closest val/test -> train pairs, shown regardless of the threshold, for manual review.
REVIEW_PAIRS = 24
PAIRS_PER_ROW = 4

image_path = dict(zip(split_df["filepath"], split_df["image_path"]))
review = nearest.head(REVIEW_PAIRS)
rows = -(-len(review) // PAIRS_PER_ROW)
fig, axes = plt.subplots(rows, PAIRS_PER_ROW * 2, figsize=(PAIRS_PER_ROW * 4, rows * 2.4))
axes = axes.reshape(rows, PAIRS_PER_ROW * 2)
for ax in axes.flat:
    ax.axis("off")

for k, pair in enumerate(review.itertuples()):
    r, c = divmod(k, PAIRS_PER_ROW)
    sides = ((pair.filepath_a, pair.split_a, pair.class_a, f" | d={pair.distance}"),
             (pair.filepath_b, pair.split_b, pair.class_b, ""))
    for offset, (fp, split, cls, extra) in enumerate(sides):
        ax = axes[r, 2 * c + offset]
        with Image.open(image_path[fp]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(f"{split} | {cls}\n{Path(fp).name}{extra}", fontsize=7)

plt.tight_layout()
plt.savefig(paths.audit_dir / "duplicate_review_grid.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

exact_groups.to_csv(paths.audit_dir / "exact_duplicate_groups.csv", index=False)
near_pairs.to_csv(paths.audit_dir / "near_duplicate_pairs.csv", index=False)
nearest.to_csv(paths.audit_dir / "nearest_train_neighbors.csv", index=False)

# dataset_summary.csv (DATA_PROTOCOL Gate 1 artifact)
folder_counts = pd.read_csv(paths.audit_dir / "folder_image_counts.csv")
files_found = int(folder_counts["num_images_found"].sum())
class_counts = split_df["class_name"].value_counts()
split_counts = split_df["split"].value_counts()

summary = {
    "image_files_found": files_found,
    "valid_images": len(split_df),
    "corrupted_images": files_found - len(split_df),
    "num_classes": split_df["class_name"].nunique(),
    "modes": ";".join(f"{k}={v}" for k, v in split_df["mode"].value_counts().items()),
    "extensions": ";".join(f"{k}={v}" for k, v in split_df["extension"].value_counts().items()),
    "dimensions": ";".join(f"{w}x{h}={n}" for (w, h), n in split_df.groupby(["width", "height"]).size().items()),
    "largest_class": f"{class_counts.idxmax()}={class_counts.max()}",
    "smallest_class": f"{class_counts.idxmin()}={class_counts.min()}",
    "imbalance_ratio": round(class_counts.max() / class_counts.min(), 2),
    "train_images": int(split_counts["train"]),
    "val_images": int(split_counts["val"]),
    "test_images": int(split_counts["test"]),
    "split_sha256": cfg["split"]["split_sha256"],
    "class_mapping_sha256": cfg["split"]["class_mapping_sha256"],
    "exact_duplicate_groups": len(exact_groups),
    "exact_duplicate_groups_cross_split": int(exact_groups["cross_split"].sum()),
    "exact_duplicate_groups_cross_class": int(exact_groups["cross_class"].sum()),
    "near_duplicate_max_distance": NEAR_DUP_MAX_DISTANCE,
    "near_duplicate_pairs": len(near_pairs),
    "near_duplicate_pairs_cross_split": int(near_pairs["cross_split"].sum()),
    "near_duplicate_pairs_cross_class": int(near_pairs["cross_class"].sum()),
    "val_test_images_with_near_train_copy": n_close,
}
summary_df = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
summary_df.to_csv(paths.audit_dir / "dataset_summary.csv", index=False)
print(summary_df.to_string(index=False))
print("\nSaved to:", paths.audit_dir)

## Manual review and decision (team)
1. Look at `duplicate_review_grid.png`: the 24 val/test images closest to a training image. For each pair decide: **same photo**, **same object re-shot**, or **only similar background/lighting**.
2. Check every `near_duplicate_pairs.csv` row with `cross_split = True` that is not already in the grid.
3. Add a new row to `docs/DECISION_LOG.md` (template printed by the next cell) and update `report/dataset_audit_notes.md`.

Suggested rule (team to confirm):
- **0 exact duplicates across splits or classes, and no confirmed same-photo pairs across splits** → keep the frozen split; Gate 1 PASS.
- **Any exact duplicate across splits/classes, or confirmed copies across splits** → do not train. Bring the list to the team/supervisor (DATA_PROTOCOL duplicate policy). A fix means removing/merging duplicates and regenerating the split under a new DECISION_LOG entry that supersedes D007.
- Same object photographed from a different angle cannot be detected here; state this as a limitation in the report.

In [ ]:
print(
    f"| D0__ | <date> | Duplicate check (03_duplicate_check.ipynb): {len(exact_groups)} exact-duplicate groups "
    f"({int(exact_groups['cross_split'].sum())} cross-split, {int(exact_groups['cross_class'].sum())} cross-class); "
    f"dHash<={NEAR_DUP_MAX_DISTANCE}: {len(near_pairs)} candidate pairs "
    f"({int(near_pairs['cross_split'].sum())} cross-split); manual review of {REVIEW_PAIRS} closest val/test-train pairs: <RESULT> "
    f"| Gate 1 duplicate requirement (PROJECT_SPEC §6) | dataset_summary.csv, duplicate_review_grid.png "
    f"| Team | <Keep split D007 / Escalate> |"
)